# Ejercicio: comparar los modos de salida

Antes de ejecutar, abre una terminal y deja corriendo:

```
nc -lk 9999
```

Los resultados del sink `console` se imprimen en la terminal desde donde lanzaste Jupyter.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split

spark = SparkSession.builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

## Función para iniciar la consulta con un modo de salida

In [ ]:
def iniciar(modo):
    return cantidadDF.writeStream \
        .format("console") \
        .outputMode(modo) \
        .start()

## Modo `complete`

Escribe en `nc` las frases: `Esta es la primera frase` y `Enviamos segunda linea primera`. Observa la salida en la terminal de Jupyter y después detén la consulta.

In [ ]:
q = iniciar("complete")

In [ ]:
q.stop()

## Modo `update`

Vuelve a enviar las mismas dos frases. Solo deben aparecer las filas que cambiaron en cada micro-batch.

In [ ]:
q = iniciar("update")

In [ ]:
q.stop()

## Modo `append`

Con una agregación sin watermark, Spark no lo permite y lanza una excepción.

In [ ]:
from pyspark.errors import AnalysisException

try:
    q = iniciar("append")
    q.stop()
except AnalysisException as e:
    print(str(e).splitlines()[0])

## Conclusión

- `complete`: en cada micro-batch muestra la tabla completa de conteos.
- `update`: muestra solo las palabras nuevas o cuyo conteo cambió.
- `append`: no es válido con agregaciones sin watermark, porque el conteo de una palabra puede cambiar con datos futuros.

In [ ]:
spark.stop()